[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/relativity/curvature/curvature.ipynb)

# Gravitational Wave Curvature & Tidal Forces in Spacetime Algebra

A passing gravitational wave stretches a ring of freely falling beads one way and squeezes it the other. In general relativity that tide is read off the Riemann curvature, a linear map on the six-dimensional space of spacetime bivectors.

In Spacetime Algebra that map is an **extensor**, `Bivector <- Bivector`, assembled from null dyads of the wave direction. For a vacuum plane wave it is nonzero yet squares to zero: all six of its eigenvalues vanish. Binding an observer's 4-velocity turns it into the **tidal map** `Vector <- Vector`, whose eigenvalues are the stretch and squeeze that observer measures. Batching over time and polarization, and broadcasting over the beads, gives the whole detector response in a few lines.


In [ ]:
# The repository root on the path, for numga and the examples; in Colab, fetch the repository first.
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    root = Path("/content/numga")
    if not root.exists():
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/EelcoHoogendoorn/numga.git", str(root)], check=True)
else:
    root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "numga").is_dir() and (p / "examples").is_dir())
sys.path.insert(0, str(root))

In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display

from numga import NumpyContext, stack
from numga.algebras import STA
from examples.animation import save_animation
from examples.relativity.curvature import render
from examples.relativity.curvature.core import wave_packet, detector_ring, integrate_acceleration

# Bind Spacetime Algebra STA: R_{1,3} with signature (t+, x-, y-, z-):
ctx = NumpyContext(STA)
mv = ctx.multivector

# Blade Subspaces:
Scalar = STA.gatype.scalar()                 # Grade 0: scalar real values
Vector = STA.gatype.vector()                 # Grade 1: events, velocities, separations
Bivector = STA.gatype.bivector()             # Grade 2: oriented spacetime areas, Lorentz generators
Rotor = STA.gatype.rotor()                   # Even subalgebra: rotations and boosts

# Extensors (linear maps between blade subspaces), read output <- input:
Curvature = STA.gatype((Bivector, Bivector))  # curvature bivector <- area bivector
Tidal = STA.gatype((Vector, Vector))          # relative acceleration <- separation
Strain = STA.gatype((Vector, Vector))         # displacement <- rest separation
Form = STA.gatype((Scalar, Vector, Vector))   # scalar <- (vector, vector): bilinear forms

# Canonical spacetime basis:
t, x, y, z = mv.vector(np.eye(4))

np.set_printoptions(precision=6, suppress=True)


## 1. Curvature from Null Dyads

A plane wave travels along the lightlike direction $k = t + z$. The bivectors $k \wedge x$ and $k \wedge y$ are null planes: each has zero magnitude, and they are orthogonal to each other. A dyad `n * (n | Bivector)` maps any bivector to $n$ scaled by its overlap with $n$, so a sum of such dyads with the area left open is a curvature extensor. Two dyads with opposite weights make the Ricci contraction vanish: this is vacuum curvature.

Because the outputs are null and mutually orthogonal, the map annihilates its own image. All six eigenvalues are zero, yet the map is not.


In [ ]:
# 1. Null wave direction and the two null planes it spans with the transverse axes:
k = t + z                                                          # [] Vector (k | k = 0)
nx, ny = k.wedge(x), k.wedge(y)                                    # [] Bivector (null planes)

# 2. Curvature as a sum of dyads. `nx | Bivector` leaves the area open: a map Scalar <- Bivector,
#    the overlap of any area with nx. Multiplying nx by that overlap gives a map Bivector <- Bivector
#    that sends every area onto nx, scaled by how much of nx it contains. Opposite weights on the
#    two dyads cancel the Ricci contraction: vacuum curvature.
plus: Curvature = nx * (nx | Bivector) - ny * (ny | Bivector)      # [] Bivector <- Bivector

# 3. The cross polarization is the plus pattern turned by 45 degrees about the wave axis.
#    A rotor conjugates a map by transforming its input and its output: rotor >> plus(rotor << Bivector)
eighth_turn: Rotor = (mv.xy * (np.pi / 8)).exp()                   # [] Rotor (half-angle: a 45 degree turn)
cross: Curvature = eighth_turn >> plus(eighth_turn << Bivector)    # [] Bivector <- Bivector

# 4. Nonzero, yet nilpotent: the image of plus consists of null planes containing k, which plus annihilates:
singular_values: Scalar = plus.svdvals()                           # [6] Scalar
# Calling a map on a map composes them: plus(plus) applies plus twice, and is the zero map:
singular_values_squared: Scalar = plus(plus).svdvals()             # [6] Scalar
eigenvalues: Scalar = plus.eigvals()                               # [6] Scalar (complex)

print("singular values of plus       :", singular_values.to_array())
print("singular values of plus(plus) :", singular_values_squared.to_array())
print("eigenvalues of plus           :", eigenvalues.to_array().real)


## 2. Riemann Symmetries as Extensor Identities

The Riemann tensor's index symmetries become statements about one map. Pair symmetry says the curvature is self-adjoint under the bivector inner product. The first Bianchi identity is a cyclic sum of contractions; with the third vector left open it is a map that must vanish. Vacuum means the Ricci form vanishes: leave the observer and separation open, and the contraction is a trace of the curvature against its wedge slot. No frame and no reciprocal basis are needed anywhere.


In [ ]:
# Random spacetime vectors and two area elements built from them:
a, b, c, d = mv.vector(np.random.default_rng(7).normal(size=(4, 4)))       # [] Vector each
ab, cd = a.wedge(b), c.wedge(d)                                             # [] Bivector

# 1. Pair symmetry: the curvature is self-adjoint under the bivector inner product:
left: Scalar = ab | plus(cd)                                                # [] Scalar
right: Scalar = plus(ab) | cd                                               # [] Scalar
pair: Scalar = stack((left, right))                                         # [2] Scalar

# 2. First Bianchi identity, with the third vector left open: the cyclic sum is the zero map:
bianchi = plus(ab).commutator(Vector) + plus(b.wedge(Vector)).commutator(a) + plus(Vector.wedge(a)).commutator(b)  # [] Vector <- Vector
bianchi_singular_values: Scalar = bianchi.svdvals()                         # [4] Scalar

# 3. Vacuum: the Ricci form, with observer and separation open. Vector.commutator(...) is the inner
#    product with an open vector; tracing the output against the wedge slot contracts the rest.
#    Two bare `Vector`s leave two slots open, so ricci is a form: it takes two vectors and returns
#    a scalar. Calling it with both, ricci(a, b), evaluates it. In index notation, R_bd = R^a_bad:
ricci: Form = Vector.commutator(plus(Vector.wedge(Vector))).trace(slot=1)   # [] Scalar <- (Vector, Vector)
ricci_on_samples: Scalar = stack((ricci(a, b), ricci(c, d), ricci(a, a)))   # [3] Scalar: zero on every pair

print("pair symmetry          :", pair.to_array())
print("Bianchi singular values:", bianchi_singular_values.to_array())
print("Ricci on sample vectors:", ricci_on_samples.to_array())
print(ricci)


## 3. The Observer's Tidal Map

Fix an observer with 4-velocity $t$. Wedge a neighbouring bead's separation with $t$ to sweep it into a small spacetime ribbon, apply the curvature, and let the resulting bivector act on $t$ again. With the separation left open this is the tidal map, separation → relative acceleration. It composes different maps rather than conjugating one, so its eigenvalues need not be those of the curvature: $+A$ along $x$, $-A$ along $y$, and zero along the wave and along time.


In [ ]:
# 1. Bind the observer twice and leave the separation open (Vector <- Vector):
#    t.wedge(Vector) sweeps a separation into a ribbon, plus(...) maps the ribbon to a null plane,
#    and .commutator(t) contracts that plane with the observer into a relative acceleration.
#    In index notation this is the geodesic deviation equation, D²ξ^a/dτ² = -R^a_bcd u^b ξ^c u^d,
#    with u = t and ξ the open separation (the overall sign depends on the curvature convention).
tidal_plus: Tidal = plus(t.wedge(Vector)).commutator(t)              # [] Vector <- Vector
tidal_eigenvalues: Scalar = tidal_plus.eigvals()                     # [4] Scalar (complex)

print("tidal eigenvalues:", tidal_eigenvalues.to_array().real)

# 2. The same composition, drawn: the ribbon t ^ x, its image (a null plane), and that image's image (zero):
fig = render.draw_curvature_map(plus, observer=t, wave=k, edge=x)
plt.show()


## 4. Boosted Observers: the Same Wave, Doppler Shifted

An observer chasing the wave sees it redshifted; one flying into it sees it blueshifted. Curvature has two time slots, so the tidal amplitude scales with the frequency squared: $e^{-2\zeta}$ for rapidity $\zeta$ along the wave. Boost rotors give a whole batch of observers at once, and binding them to the curvature gives a batch of tidal maps whose singular values are that amplitude.


In [ ]:
# 1. Observers boosted along the wave direction, batched over rapidity:
rapidities = np.linspace(-0.7, 0.7, 15)
boosts: Rotor = ((z ^ t) * (rapidities / 2)).exp()                     # [n_obs] Rotor
observers: Vector = boosts >> t                                         # [n_obs] Vector (4-velocities)

# 2. Bind each observer to the same curvature; the batch axis carries through the composition:
responses: Tidal = plus(observers.wedge(Vector)).commutator(observers)  # [n_obs] Vector <- Vector
amplitudes: Scalar = responses.svdvals()[..., 0]                        # [n_obs] Scalar (largest singular value)

fig = render.draw_doppler(rapidities, amplitudes)
plt.show()


## 5. A Wave Packet in Three Polarizations

For a weak wave the curvature is $R_{0i0j} = -\tfrac{1}{2}\ddot h_{ij}$: the second derivative of the strain weights the unit maps. A $\sin^4$ envelope with three carrier cycles starts and ends at rest. Scaling `plus` by the cosine profile and `cross` by the sine profile, then stacking plus, cross, and their sum, gives one curvature batch over time and polarization.


In [ ]:
# 1. Strain profiles and their second derivatives as scalar batches over (time, phase):
time = np.linspace(0.0, 6.0, 1201)
strain, second = wave_packet(time, duration=6.0, cycles=3, amplitude=1e-4)   # [n_time, 2] Scalar
cosine, sine = second[:, 0], second[:, 1]                                    # [n_time] Scalar

# 2. Multiplying a map by a batch of scalars gives a batch of maps, one per time step; stacking
#    three such batches adds a polarization axis. Weak-wave curvature over (time, polarization):
plus_wave, cross_wave = plus * cosine, cross * sine                          # [n_time] Bivector <- Bivector
waves: Curvature = -0.5 * stack((plus_wave, cross_wave, plus_wave + cross_wave), axis=1)  # [n_time, 3] Bivector <- Bivector

fig = render.draw_packet(time, strain, second)
plt.show()


## 6. A Ring of Freely Falling Beads

Bind the observer to the whole batch and the tidal map follows along: `[n_time, 3] Vector <- Vector`. Applying it to a ring of reference separations broadcasts over the beads, giving every bead's relative acceleration at every time for every polarization. Integrating twice from rest gives the displacements. The detector is much smaller than the wavelength, so the acceleration acts on each bead's unperturbed separation.


In [ ]:
# 1. The observer's tidal map for the whole packet:
response: Tidal = waves(t.wedge(Vector)).commutator(t)                 # [n_time, 3] Vector <- Vector

# 2. A ring of beads at radius 0.01, broadcast against the batch of tidal maps:
reference: Vector = detector_ring(24) * 0.01                           # [n_beads] Vector
acceleration: Vector = response[:, :, None](reference)                 # [n_time, 3, n_beads] Vector

# 3. Integrate twice from rest (numerical boundary):
displacement: Vector = integrate_acceleration(time, acceleration)      # [n_time, 3, n_beads] Vector

# Plot: the three polarizations at one shared instant, displacements magnified for display:
fig = render.draw_detector(time, reference, displacement, acceleration, amplification=4000)
plt.show()


## 7. The Same Wave as a Strain Map

In gauge theory gravity the wave is carried by a gauge field rather than a metric: a map on vectors that differs from the identity by a strain map, whose value on a rest separation is the physical separation. For the plus polarization the strain stretches along $x$ and squeezes along $y$, and the cross pattern is the same map turned by 45 degrees, as the curvature was. Half the metric perturbation is the strain, so each bead's displacement is the strain map applied to its rest separation, with no integration.

The curvature is the strain's second derivative along the wave. Wedge the wave vector into that second derivative and weight by the overlap with the wave vector: the result is the curvature as a map on pairs of vectors, the two edges of the area, and it agrees with the dyad construction of section 1 at every time and polarization.

In [ ]:
# 1. Unit strain patterns on separations: stretch along x and squeeze along y, and the same turned by 45 degrees:
plus_strain: Strain = y * (y | Vector) - x * (x | Vector)                    # [] Vector <- Vector
cross_strain: Strain = eighth_turn >> plus_strain(eighth_turn << Vector)     # [] Vector <- Vector

# 2. Half the metric perturbation as a map on separations, batched over time and stacked over polarization:
cosine_strain, sine_strain = strain[:, 0], strain[:, 1]                      # [n_time] Scalar
plus_wave_strain, cross_wave_strain = plus_strain * cosine_strain, cross_strain * sine_strain
strain_map: Strain = 0.5 * stack((plus_wave_strain, cross_wave_strain, plus_wave_strain + cross_wave_strain), axis=1)  # [n_time, 3] Vector <- Vector

# 3. Each bead's displacement is the strain map applied to its rest separation, and the integrated ring lands on it:
predicted: Vector = strain_map[:, :, None](reference)                        # [n_time, 3, n_beads] Vector
mismatch: Scalar = ((displacement - predicted) | (displacement - predicted)).abs().square_root()   # [n_time, 3, n_beads] Scalar
peak: Scalar = (predicted | predicted).abs().square_root()                   # [n_time, 3, n_beads] Scalar

# 4. Curvature from the strain's second derivative. Each factor carries its own open Vector,
#    k.wedge(second_strain) and (k | Vector), so their product has two slots, in the order they
#    appear: a map from the two edges of an area to a bivector, Bivector <- (Vector, Vector).
second_strain: Strain = 0.5 * stack((plus_strain * cosine, cross_strain * sine, plus_strain * cosine + cross_strain * sine), axis=1)  # [n_time, 3] Vector <- Vector
curvature_two_form = k.wedge(second_strain) * (k | Vector) - (k | Vector) * k.wedge(second_strain)   # [n_time, 3] Bivector <- (Vector, Vector)

# 5. Calling it with one edge binds the first slot and leaves the second open: Bivector <- Vector.
#    Compare with the dyad curvature applied to the wedge with that edge:
edge: Vector = mv.vector(np.random.default_rng(5).normal(size=4))            # [] Vector
from_strain = curvature_two_form(edge)                                  # [n_time, 3] Bivector <- Vector
from_dyads = waves(edge.wedge(Vector))                                       # [n_time, 3] Bivector <- Vector
agreement: Scalar = (from_strain - from_dyads).svdvals()                     # [n_time, 3, 4] Scalar

print("max |integrated - strain(reference)| :", mismatch.to_array().max())
print("peak displacement                    :", peak.to_array().max())
print("curvature from strain vs from dyads  :", agreement.to_array().max())
print(curvature_two_form)

## 8. Animation

Released from rest, the ring returns to rest once the packet has passed, to first order. In the circular case the elliptical pattern turns while each coloured bead traces a small loop about its rest position: a turning deformation, not a rigid rotation.


In [ ]:
frames = render.animate_detector(time, reference, displacement, acceleration, amplification=4000)
gif = save_animation(frames, "curvature", 50)
display(Image(filename=gif))

## 9. Summary

* **Curvature as a dyad sum**: `nx * (nx | Bivector)` with the area left open builds the Riemann map of a plane wave; opposite weights on the two null dyads make it vacuum.
* **Polarization by conjugation**: `rotor >> plus(rotor << Bivector)` turns the plus pattern into cross, the same double-sided transport that boosts a material or moves a camera.
* **Nilpotent but not zero**: the null outputs lie in the map's own kernel, so all six eigenvalues vanish. The indefinite metric permits this for a self-adjoint map.
* **Observer binding is composition, not conjugation**: `curvature(t.wedge(Vector)).commutator(t)` has eigenvalues $\pm A$ although the curvature has none, and a boosted observer sees them scaled by $e^{-2\zeta}$.
* **Batches carry through**: time, polarization and observer rapidity are batch axes on the maps; beads broadcast against them, so the whole detector response is one expression.

* **The same wave as a strain map**: in gauge theory gravity the wave is a map on vectors; half the metric perturbation applied to a bead's rest separation is its displacement, and the strain's second derivative wedged with the wave vector is the curvature, `k.wedge(second) * (k | Vector) - (k | Vector) * k.wedge(second)`.